# Part 3: µP Experiments (LR Sweep + Scaling Study)

SVG Scaling Laws Project

**Runtime**: A100 GPU

**Prerequisites**: Run `colab_lr_sweep.ipynb` first (data preprocessing).

**Sections**:
1. Setup
2. µP LR Sweep (Tiny, 7 LRs) — ~20 min
3. µP Scaling Study (all 5 models) — ~2.5 hours

**Total estimated time**: ~3 hours

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify data exists
import os
assert os.path.exists('data/tokenized/train.bin'), 'Run colab_lr_sweep.ipynb first!'

## 2. µP LR Sweep (Tiny, 7 LRs)

Higher LRs (1e-2, 3e-2) included because µP should stabilize them.

In [ ]:
import subprocess, time, json

learning_rates = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3', '1e-2', '3e-2']
sweep_results = []

for lr in learning_rates:
    output_dir = f'results/runs/mup_lr_sweep/tiny_lr_{lr}'
    print(f'\n{"=" * 60}')
    print(f'\u00b5P LR Sweep: Tiny, LR={lr}')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', 'src/train.py',
        '--config', 'configs/tiny.yaml',
        '--learning-rate', lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
        '--mup',
    ])

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n\u2192 LR={lr}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        sweep_results.append({'lr': lr, **s})
        print(f'  final_val_loss={s["final_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

In [ ]:
# Determine optimal LR (by final_val_loss)
print(f'{"LR":>8s}  {"Final Val Loss":>14s}  {"Best Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 52)
for r in sorted(sweep_results, key=lambda x: x['final_val_loss']):
    print(f'{r["lr"]:>8s}  {r["final_val_loss"]:>14.4f}  {r["best_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')

best = min(sweep_results, key=lambda x: x['final_val_loss'])
optimal_lr = best['lr']
print(f'\n\u2192 Optimal \u00b5P LR: {optimal_lr} (final_val_loss={best["final_val_loss"]:.4f})')

## 3. µP Scaling Study (All 5 Models)

Train all 5 models with the optimal µP LR found above.

Estimated time: ~2.5 hours (Tiny 5m + Small 10m + Medium 20m + Large 30m + XL 80m)

In [ ]:
models = ['tiny', 'small', 'medium', 'large', 'xl']
scaling_results = []

for name in models:
    output_dir = f'results/runs/mup/{name}'
    print(f'\n{"=" * 60}')
    print(f'\u00b5P Scaling Study: {name} (LR={optimal_lr})')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', 'src/train.py',
        '--config', f'configs/{name}.yaml',
        '--learning-rate', optimal_lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
        '--mup',
    ])

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n\u2192 {name}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        scaling_results.append(s)
        print(f'  final_val_loss={s["final_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

## 4. Results

In [ ]:
print(f'{"Model":>8s}  {"Params":>8s}  {"Final Val Loss":>14s}  {"Best Val Loss":>14s}  {"Final PPL":>10s}')
print('-' * 60)
for r in scaling_results:
    print(f'{r["config_name"]:>8s}  {r["n_params"]/1e6:>7.1f}M  '
          f'{r["final_val_loss"]:>14.4f}  {r["best_val_loss"]:>14.4f}  '
          f'{r["final_val_ppl"]:>10.2f}')

print(f'\n\u00b5P Scaling Study complete. Run analyze_mup.py locally for power law fit + plots.')